### 🏷️ **Credits & License**

* 🔗 [OmniVoice GitHub Repository](https://github.com/k2-fsa/OmniVoice)
* 🤗 [OmniVoice on Hugging Face](https://huggingface.co/k2-fsa/OmniVoice)
* 📄 **License**: Provided under the [Apache License 2.0](https://github.com/k2-fsa/OmniVoice/blob/master/LICENSE)



### ⚠️ **Usage Disclaimer**

Use of this voice cloning model is subject to strict ethical and legal standards. By using this tool, you agree **not to** engage in any of the following prohibited activities:

* **Fraud or Deception**: Using cloned voices to create misleading or fraudulent content.
* **Impersonation**: Replicating someone's voice without their explicit permission, especially for malicious, harmful, or deceptive purposes.
* **Illegal Activities**: Employing the model in any manner that violates local, national, or international laws and regulations.
* **Harmful Content Generation**: Creating offensive, defamatory, or unethical material, including content that spreads misinformation or causes harm.

> ⚖️ **Legal Responsibility**
> The developers of this tool disclaim all liability for misuse. **Users bear full responsibility** for ensuring that their usage complies with all applicable laws, regulations, and ethical guidelines.


In [ ]:
#@title Install OmniVoice
%cd /content/
!rm -rf ./omnivoice-colab
!git clone https://github.com/hirannalaka19/omnivoice-colab.git
%cd ./omnivoice-colab
# OmniVoice is now an official pip package: https://github.com/k2-fsa/OmniVoice
!pip install -r colab.txt
from IPython.display import clear_output
clear_output()
print("✅ Installation complete")

In [ ]:
#@title Run Gradio APP
#@markdown Uses the `HF_TOKEN` saved in Colab **Secrets** (🔑 icon in the left sidebar): add a secret named `HF_TOKEN` with your Hugging Face token (create one at https://huggingface.co/settings/tokens, type: **Read**) and enable **Notebook access**. One-time setup — it persists across all sessions.

import os, subprocess, sys, time

HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    HF_TOKEN = ""

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("🔑 Using HF_TOKEN from Colab Secrets — full download speed")
else:
    print("⚠️ No HF_TOKEN secret found — add it via the 🔑 Secrets sidebar (name: HF_TOKEN) and enable Notebook access. Anonymous downloads are throttled/blocked.")

# Hugging Face's CDN intermittently rejects its own signed URLs with
# 403 "invalid key pair id" (a server-side glitch). Already-downloaded files
# are cached, so retrying only fetches what is still missing — we retry the
# fast HTTP path several times, then fall back to the Xet protocol path.
def try_download(disable_xet):
    env = dict(os.environ)
    if disable_xet:
        env["HF_HUB_DISABLE_XET"] = "1"
    else:
        env.pop("HF_HUB_DISABLE_XET", None)
    return subprocess.call(
        [sys.executable, "-c",
         "from huggingface_hub import snapshot_download; snapshot_download('k2-fsa/OmniVoice')"],
        env=env,
    ) == 0

ok = False
for attempt in range(1, 5):
    print(f"⬇️ Downloading the model — fast HTTP path (attempt {attempt}/4) ...")
    if try_download(disable_xet=True):
        ok = True
        break
    print("⚠️ Hit the Hugging Face CDN glitch — retrying (already-downloaded files are kept) ...")
    time.sleep(5)

if not ok:
    for attempt in range(1, 3):
        print(f"⬇️ Falling back to the Xet protocol path (attempt {attempt}/2, can be slower) ...")
        if try_download(disable_xet=False):
            ok = True
            break
        time.sleep(5)

if not ok:
    raise SystemExit("❌ All download attempts failed — Hugging Face may be having an outage (check https://status.huggingface.co). Re-run this cell to retry.")

print("✅ Model downloaded")

# Model is now cached locally, so the app starts without re-downloading.
os.environ["HF_HUB_DISABLE_XET"] = "1"

%cd /content/omnivoice-colab
!python app.py